In [20]:
%pip install kagglehub

import kagglehub
import pandas as pd
import os

print("Localizando arquivos...")
path_stats = kagglehub.dataset_download("hubertsidorowicz/football-players-stats-2025-2026")

file_name = 'players_data_light-2025_2026.csv'
full_path = os.path.join(path_stats, file_name)

if os.path.exists(full_path):
    df = pd.read_csv(full_path)
    
    print("\n" + "="*50)
    print("ESTRUTURA DA TABELA DE JOGADORES")
    print("="*50)
    
    print(df.info())
    
    print("\n--- VISUALIZAÇÃO DOS DADOS (HEAD) ---")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(df.head())
    
else:
    print(f"Erro: O arquivo {file_name} não foi encontrado na pasta {path_stats}")
    print("Arquivos presentes na pasta:", os.listdir(path_stats))


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Localizando arquivos...

ESTRUTURA DA TABELA DE JOGADORES
<class 'pandas.DataFrame'>
RangeIndex: 2718 entries, 0 to 2717
Data columns (total 53 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Rk                    2718 non-null   int64  
 1   Player                2718 non-null   str    
 2   Nation                2715 non-null   str    
 3   Pos                   2718 non-null   str    
 4   Squad                 2718 non-null   str    
 5   Comp                  2718 non-null   str    
 6   Age                   2715 non-null   float64
 7   Born                  2715 non-null   float64
 8   MP                    2718 non-null   int64  
 9   Starts                2718 non-null   int64  
 10  Min                   2718 non-null   int64  
 11  90s                   2718 non-null   float64
 12  Gls                   2718 non-null   int64  
 13  Ast       

## Análise de Valores Ausentes (NaN)

Abaixo, vamos contar a quantidade de valores `NaN` em cada coluna para entender a extensão dos dados faltantes.

In [4]:
import pandas as pd
import numpy as np

# Supondo que 'df' já está carregado (players_data_light-2025_2026.csv)

print("--- CONTAGEM INICIAL DE VALORES AUSENTES (NaN) POR COLUNA ---")
nan_counts = df.isnull().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(nan_counts)

# =============================================================================
# TRATAMENTO DOS NaN - LÓGICA RECOMENDADA PARA O PROJETO
# =============================================================================

# 1. Colunas de SHOOTING (ataque): NaN significa "zero eventos" → preenche com 0
shooting_cols = ['G/SoT', 'SoT%', 'G/Sh']
# Se essas colunas existirem no seu df
shooting_cols = [col for col in shooting_cols if col in df.columns]
df[shooting_cols] = df[shooting_cols].fillna(0)

# 2. Colunas de GOLEIROS (keeper stats): NaN para não-goleiros → manter NaN ou 0?
# Aqui colocamos 0 só para não-goleiros (pra não poluir médias), mas mantemos NaN pros goleiros se faltar dado
keeper_cols = ['CS%', 'Save%', 'GA90', 'SoTA', 'PKatt_stats_keeper', 'PKA', 'L', 'CS', 
               'Saves', 'W', 'D', 'GA', 'PKsv', 'PKm']
keeper_cols = [col for col in keeper_cols if col in df.columns]

# Preenche com 0 SOMENTE se NÃO for goleiro
for col in keeper_cols:
    df[col] = df.apply(
        lambda row: 0 if (pd.isna(row[col]) and 'GK' not in str(row.get('Pos', ''))) 
                    else row[col],
        axis=1
    )

# Opcional: forçar 0 nos goleiros que tiverem NaN (raro, mas por segurança)
df[keeper_cols] = df[keeper_cols].fillna(0)

# 3. Colunas básicas (Age, Nation, Born): poucas linhas → drop ou preencher com mediana/moda
basic_cols = ['Age', 'Nation', 'Born']
basic_cols = [col for col in basic_cols if col in df.columns]

# Exemplo: preencher Age com mediana, Nation com 'Unknown'
if 'Age' in df.columns:
    df['Age'] = df['Age'].fillna(df['Age'].median())
if 'Nation' in df.columns:
    df['Nation'] = df['Nation'].fillna('Unknown')
if 'Born' in df.columns:
    df['Born'] = df['Born'].fillna(df['Born'].median())

# =============================================================================
# FEATURES ÚTEIS PARA O DESAFIO (enriquecimento live)
# =============================================================================

# Gols + Assists por 90 minutos (métrica chave pra comparar jogadores)
if all(col in df.columns for col in ['G+A', '90s']):
    df['G+A_per90'] = (df['G+A'] / df['90s'].replace(0, np.nan)).round(2)

# Exemplo extra: Gols esperados por 90 (se tiver xG na light - verifique!)
if 'xG' in df.columns and '90s' in df.columns:
    df['xG_per90'] = (df['xG'] / df['90s'].replace(0, np.nan)).round(2)

# =============================================================================
# VERIFICAÇÃO FINAL
# =============================================================================

print("\n" + "="*60)
print("NOVOS NaN APÓS TRATAMENTO (deve ser bem menos ou zero nas cols principais)")
print("="*60)
final_nan = df.isnull().sum()
final_nan = final_nan[final_nan > 0].sort_values(ascending=False)
if final_nan.empty:
    print("Nenhum NaN restante nas colunas! Perfeito.")
else:
    print(final_nan)

# Amostra do df tratado (pra você ver se ficou bom)
print("\nAmostra do DataFrame tratado (primeiras 8 linhas):")
print(df[['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', '90s', 'Gls', 'Ast', 'G+A', 'G+A_per90',
          'G/SoT', 'SoT%', 'Save%', 'GA90']].head(8))

# Salvar o df limpo (opcional, mas recomendado pro resto do hack)
# df.to_csv('players_data_light_clean_2025_2026.csv', index=False)
# print("\nDataFrame limpo salvo como 'players_data_light_clean_2025_2026.csv'")

--- CONTAGEM INICIAL DE VALORES AUSENTES (NaN) POR COLUNA ---
CS%                   2551
Save%                 2549
GA90                  2548
SoTA                  2548
PKatt_stats_keeper    2548
PKA                   2548
L                     2548
CS                    2548
Saves                 2548
W                     2548
D                     2548
GA                    2548
PKsv                  2548
PKm                   2548
G/SoT                  852
SoT%                   464
G/Sh                   464
Born                     3
Nation                   3
Age                      3
dtype: int64

NOVOS NaN APÓS TRATAMENTO (deve ser bem menos ou zero nas cols principais)
G+A_per90    52
dtype: int64

Amostra do DataFrame tratado (primeiras 8 linhas):
                Player   Nation    Pos              Squad                Comp   Age   90s  Gls  Ast  G+A  G+A_per90  G/SoT  SoT%  Save%  GA90
0     Brenden Aaronson   us USA  MF,FW       Leeds United  eng Premier League  25.0  2

## Análise do 2º Dataset (StatsBomb)

Agora vamos carregar o banco de eventos do StatsBomb (`events.csv`) e entender sua estrutura para extrair insights de jogo.

In [5]:
# Carregar dados de eventos do StatsBomb (CSV ou JSON)
import glob
import json

path_sb = kagglehub.dataset_download("saurabhshahane/statsbomb-football-data")
file_sb = os.path.join(path_sb, "events.csv")

if os.path.exists(file_sb):
    df_sb = pd.read_csv(file_sb)
    print("\n" + "="*60)
    print("ESTRUTURA DO DATASET STATSBOMB (EVENTOS - CSV)")
    print("="*60)
    print(f"Linhas: {df_sb.shape[0]:,} | Colunas: {df_sb.shape[1]}")
    print("\n--- AMOSTRA DOS DADOS ---")
    print(df_sb.head())
else:
    # Fallback para o formato real desta base: JSONs em data/events
    event_json_files = glob.glob(os.path.join(path_sb, "**", "events", "*.json"), recursive=True)
    print(f"events.csv não encontrado. JSONs de eventos encontrados: {len(event_json_files)}")

    if not event_json_files:
        print(f"Nenhum arquivo de eventos foi encontrado em {path_sb}")
    else:
        records = []
        max_files = 20  # amostra para carregar rápido
        for fp in event_json_files[:max_files]:
            with open(fp, "r", encoding="utf-8") as f:
                events = json.load(f)
            if isinstance(events, list):
                records.extend(events)

        if records:
            df_sb = pd.json_normalize(records)
            print("\n" + "="*60)
            print("ESTRUTURA DO DATASET STATSBOMB (EVENTOS - JSON)")
            print("="*60)
            print(f"Linhas: {df_sb.shape[0]:,} | Colunas: {df_sb.shape[1]}")
            print("\nPrimeiras colunas:")
            print(df_sb.columns[:20].tolist())
            print("\n--- AMOSTRA DOS DADOS ---")
            print(df_sb.head())
        else:
            print("JSONs encontrados, mas nenhum registro válido foi carregado.")

events.csv não encontrado. JSONs de eventos encontrados: 3464

ESTRUTURA DO DATASET STATSBOMB (EVENTOS - JSON)
Linhas: 75,951 | Colunas: 137

Primeiras colunas:
['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type.id', 'type.name', 'possession_team.id', 'possession_team.name', 'play_pattern.id', 'play_pattern.name', 'team.id', 'team.name', 'tactics.formation', 'tactics.lineup', 'related_events', 'location']

--- AMOSTRA DOS DADOS ---
                                     id  index  period     timestamp  minute  second  possession  duration  type.id    type.name  possession_team.id possession_team.name  play_pattern.id play_pattern.name  team.id         team.name  tactics.formation                                     tactics.lineup                          related_events      location  player.id                  player.name  position.id  position.name  pass.recipient.id               pass.recipient.name  pass.length  pass.angle  pass.height.id pass.h

In [6]:
# Diagnóstico: localizar automaticamente arquivos de evento no dataset StatsBomb
import glob

csv_files = glob.glob(os.path.join(path_sb, "**", "*.csv"), recursive=True)
json_files = glob.glob(os.path.join(path_sb, "**", "*.json"), recursive=True)

print("CSV encontrados:", len(csv_files))
print("JSON encontrados:", len(json_files))

# Mostrar alguns caminhos para inspeção
print("\nExemplos de CSV:")
for p in csv_files[:20]:
    print("-", p)

# Tentar achar automaticamente um arquivo de eventos
event_candidates = [p for p in csv_files if "event" in os.path.basename(p).lower() or "events" in p.lower()]

if event_candidates:
    chosen_file = event_candidates[0]
    print(f"\nArquivo de eventos escolhido automaticamente: {chosen_file}")
    df_sb = pd.read_csv(chosen_file)
    print(f"df_sb carregado com sucesso -> Linhas: {df_sb.shape[0]:,}, Colunas: {df_sb.shape[1]}")
else:
    print("\nNenhum CSV de eventos encontrado automaticamente.")
    print("Pode ser que este dataset esteja em JSON; nesse caso podemos montar um parser na próxima célula.")

CSV encontrados: 0
JSON encontrados: 7330

Exemplos de CSV:

Nenhum CSV de eventos encontrado automaticamente.
Pode ser que este dataset esteja em JSON; nesse caso podemos montar um parser na próxima célula.


In [7]:
# Carregar eventos a partir dos JSONs do StatsBomb
import json

# Arquivos JSON de eventos (geralmente ficam em pasta 'events')
event_json_files = [p for p in json_files if "events" in p.lower() and p.lower().endswith(".json")]

print(f"Arquivos JSON de eventos encontrados: {len(event_json_files)}")
for p in event_json_files[:5]:
    print("-", p)

# Limitar quantidade para análise inicial (rápido)
MAX_FILES = 50
selected_files = event_json_files[:MAX_FILES]

records = []
for fp in selected_files:
    try:
        with open(fp, "r", encoding="utf-8") as f:
            events = json.load(f)
        if isinstance(events, list):
            records.extend(events)
    except Exception as e:
        print(f"Falha ao ler {fp}: {e}")

if records:
    df_sb = pd.json_normalize(records)
    print(f"\ndf_sb carregado de JSON com sucesso -> Linhas: {df_sb.shape[0]:,}, Colunas: {df_sb.shape[1]}")
    print("\nColunas principais:")
    print(df_sb.columns[:20].tolist())
else:
    print("Nenhum evento foi carregado dos arquivos JSON selecionados.")

Arquivos JSON de eventos encontrados: 3464
- C:\Users\kaiki\.cache\kagglehub\datasets\saurabhshahane\statsbomb-football-data\versions\1378\data\events\15946.json
- C:\Users\kaiki\.cache\kagglehub\datasets\saurabhshahane\statsbomb-football-data\versions\1378\data\events\15956.json
- C:\Users\kaiki\.cache\kagglehub\datasets\saurabhshahane\statsbomb-football-data\versions\1378\data\events\15973.json
- C:\Users\kaiki\.cache\kagglehub\datasets\saurabhshahane\statsbomb-football-data\versions\1378\data\events\15978.json
- C:\Users\kaiki\.cache\kagglehub\datasets\saurabhshahane\statsbomb-football-data\versions\1378\data\events\15986.json

df_sb carregado de JSON com sucesso -> Linhas: 188,165, Colunas: 140

Colunas principais:
['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type.id', 'type.name', 'possession_team.id', 'possession_team.name', 'play_pattern.id', 'play_pattern.name', 'team.id', 'team.name', 'tactics.formation', 'tactics.lineup', 'related_even

In [8]:
# Insights iniciais: eventos mais frequentes, times e jogadores mais ativos
if 'df_sb' in globals():
    event_col_candidates = [c for c in df_sb.columns if 'type' in c.lower()]
    team_col_candidates = [c for c in df_sb.columns if 'team' in c.lower()]
    player_col_candidates = [c for c in df_sb.columns if 'player' in c.lower()]

    print("\nColunas candidatas:")
    print("Evento:", event_col_candidates[:8])
    print("Time:", team_col_candidates[:8])
    print("Jogador:", player_col_candidates[:8])

    # Priorizar nomes comuns em CSV e em JSON normalizado
    event_priority = ['type.name', 'type_name', 'event_type', 'type']
    team_priority = ['team.name', 'team_name', 'team', 'possession_team.name']
    player_priority = ['player.name', 'player_name', 'player']

    event_col = next((c for c in event_priority if c in df_sb.columns), None)
    team_col = next((c for c in team_priority if c in df_sb.columns), None)
    player_col = next((c for c in player_priority if c in df_sb.columns), None)

    if event_col:
        print(f"\n--- TOP 10 TIPOS DE EVENTO ({event_col}) ---")
        print(df_sb[event_col].value_counts(dropna=False).head(10))
    else:
        print("\nColuna de evento não encontrada.")

    if team_col:
        print(f"\n--- TOP 10 TIMES COM MAIS EVENTOS ({team_col}) ---")
        print(df_sb[team_col].value_counts(dropna=False).head(10))
    else:
        print("\nColuna de time não encontrada.")

    if player_col:
        print(f"\n--- TOP 10 JOGADORES COM MAIS EVENTOS ({player_col}) ---")
        print(df_sb[player_col].value_counts(dropna=False).head(10))
    else:
        print("\nColuna de jogador não encontrada.")
else:
    print("Execute a célula de carregamento do StatsBomb primeiro.")


Colunas candidatas:
Evento: ['type.id', 'type.name', 'pass.type.id', 'pass.type.name', 'duel.type.id', 'duel.type.name', 'shot.type.id', 'shot.type.name']
Time: ['possession_team.id', 'possession_team.name', 'team.id', 'team.name']
Jogador: ['player.id', 'player.name']

--- TOP 10 TIPOS DE EVENTO (type.name) ---
type.name
Pass             53570
Ball Receipt*    49617
Carry            44433
Pressure         16067
Ball Recovery     4777
Duel              2745
Block             1882
Dribble           1782
Clearance         1609
Goal Keeper       1590
Name: count, dtype: int64

--- TOP 10 TIMES COM MAIS EVENTOS (team.name) ---
team.name
Barcelona          85352
Real Madrid        10886
Atlético Madrid     7332
Bayern Munich       6565
Chelsea FCW         3952
Real Betis          3678
Real Sociedad       3416
Sevilla             3301
Reading WFC         3285
Eibar               3280
Name: count, dtype: int64

--- TOP 10 JOGADORES COM MAIS EVENTOS (player.name) ---
player.name
Lionel Andrés

In [9]:
# Síntese de insights dos dois datasets
print("="*70)
print("INSIGHTS - DATASET DE JOGADORES")
print("="*70)

print(f"Registros de jogadores: {len(df):,}")
print(f"Colunas: {df.shape[1]}")

# Valores ausentes mais relevantes
missing_top = df.isna().sum().sort_values(ascending=False)
missing_top = missing_top[missing_top > 0].head(10)
print("\nTop 10 colunas com mais NaN:")
print(missing_top)

# Tentar encontrar campos úteis para ranking
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(f"\nTotal de colunas numéricas: {len(numeric_cols)}")

for candidate in ['Gls', 'Ast', 'xG', 'xAG', 'SoT', 'Age', 'MP', 'Min']:
    if candidate in df.columns:
        print(f"\nResumo de {candidate}:")
        print(df[candidate].describe())

print("\n" + "="*70)
print("INSIGHTS - DATASET DE EVENTOS (STATSBOMB)")
print("="*70)

print(f"Registros de eventos: {len(df_sb):,}")
print(f"Colunas: {df_sb.shape[1]}")

# Colunas principais do StatsBomb normalizado
event_col = 'type.name' if 'type.name' in df_sb.columns else None
team_col = 'team.name' if 'team.name' in df_sb.columns else None
player_col = 'player.name' if 'player.name' in df_sb.columns else None
minute_col = 'minute' if 'minute' in df_sb.columns else None

if event_col:
    print("\nTop 10 tipos de evento:")
    print(df_sb[event_col].value_counts(dropna=False).head(10))

if team_col:
    print("\nTop 10 times com mais eventos:")
    print(df_sb[team_col].value_counts(dropna=False).head(10))

if player_col:
    print("\nTop 10 jogadores com mais eventos:")
    print(df_sb[player_col].value_counts(dropna=False).head(10))

if minute_col and event_col:
    # Pressão temporal simples: em quais minutos há mais eventos
    busiest_minutes = df_sb[minute_col].value_counts().sort_values(ascending=False).head(10)
    print("\nMinutos com maior volume de eventos:")
    print(busiest_minutes)

# Foco em finalização, se existir informação de chute
if event_col and 'shot.outcome.name' in df_sb.columns:
    shots = df_sb[df_sb[event_col] == 'Shot'].copy()
    print(f"\nTotal de finalizações (Shot): {len(shots):,}")
    if len(shots) > 0:
        print("Distribuição de resultado dos chutes:")
        print(shots['shot.outcome.name'].value_counts(dropna=False).head(10))


INSIGHTS - DATASET DE JOGADORES
Registros de jogadores: 2,718
Colunas: 54

Top 10 colunas com mais NaN:
G+A_per90    52
dtype: int64

Total de colunas numéricas: 49

Resumo de Gls:
count    2718.000000
mean        1.307579
std         2.341074
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max        30.000000
Name: Gls, dtype: float64

Resumo de Ast:
count    2718.000000
mean        0.905813
std         1.484292
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        17.000000
Name: Ast, dtype: float64

Resumo de SoT:
count    2718.000000
mean        4.185798
std         5.944015
min         0.000000
25%         0.000000
50%         2.000000
75%         6.000000
max        59.000000
Name: SoT, dtype: float64

Resumo de Age:
count    2718.000000
mean       25.635394
std         4.547289
min        16.000000
25%        22.000000
50%        25.000000
75%        29.000000
max        42.000000
Name: Age, dtype: flo

## Loader adaptado ao Workspace (prioriza arquivos locais)

Esta seção procura datasets no workspace primeiro. Se não encontrar, usa cache/download do Kaggle como fallback.

In [10]:
# Versao adaptada: procura dados locais no workspace antes de usar Kaggle cache/download
import os
import glob
import json
from pathlib import Path

import pandas as pd
import kagglehub

# Tenta buscar a partir da pasta atual e pastas pai (caso o notebook rode em Data/Copper)
start = Path.cwd()
search_roots = [start, start.parent, start.parent.parent]
search_roots = [p for p in search_roots if p.exists()]
print("Raizes de busca:")
for r in search_roots:
    print("-", r)

# 1) Players dataset (local-first)
player_candidates = []
for root in search_roots:
    player_candidates.extend(root.rglob("players_data_light-2025_2026.csv"))
    player_candidates.extend(root.rglob("players_data-2025_2026.csv"))

if player_candidates:
    player_path = player_candidates[0]
    print(f"[LOCAL] Players encontrado: {player_path}")
    df_players_local = pd.read_csv(player_path)
else:
    print("[LOCAL] Players nao encontrado no workspace. Usando Kaggle fallback.")
    path_stats_fallback = kagglehub.dataset_download("hubertsidorowicz/football-players-stats-2025-2026")
    player_path = Path(path_stats_fallback) / "players_data_light-2025_2026.csv"
    if not player_path.exists():
        player_path = Path(path_stats_fallback) / "players_data-2025_2026.csv"
    df_players_local = pd.read_csv(player_path)
    print(f"[FALLBACK] Players carregado de: {player_path}")

# 2) StatsBomb events (local-first)
local_event_json = []
local_event_csv = []
for root in search_roots:
    local_event_json.extend(root.rglob("events/*.json"))
    local_event_csv.extend(root.rglob("events.csv"))

if local_event_csv:
    sb_path = local_event_csv[0]
    print(f"[LOCAL] StatsBomb CSV encontrado: {sb_path}")
    df_events_local = pd.read_csv(sb_path)
elif local_event_json:
    print(f"[LOCAL] StatsBomb JSONs encontrados no workspace: {len(local_event_json)}")
    sample_files = local_event_json[:30]
    records = []
    for fp in sample_files:
        with open(fp, "r", encoding="utf-8") as f:
            events = json.load(f)
        if isinstance(events, list):
            records.extend(events)
    df_events_local = pd.json_normalize(records)
else:
    print("[LOCAL] StatsBomb nao encontrado no workspace. Usando Kaggle fallback.")
    path_sb_fallback = kagglehub.dataset_download("saurabhshahane/statsbomb-football-data")
    fallback_json = glob.glob(str(Path(path_sb_fallback) / "**" / "events" / "*.json"), recursive=True)
    fallback_csv = Path(path_sb_fallback) / "events.csv"

    if fallback_csv.exists():
        df_events_local = pd.read_csv(fallback_csv)
        print(f"[FALLBACK] StatsBomb CSV carregado de: {fallback_csv}")
    elif fallback_json:
        print(f"[FALLBACK] JSONs de eventos encontrados: {len(fallback_json)}")
        records = []
        for fp in fallback_json[:30]:
            with open(fp, "r", encoding="utf-8") as f:
                events = json.load(f)
            if isinstance(events, list):
                records.extend(events)
        df_events_local = pd.json_normalize(records)
    else:
        raise FileNotFoundError("Nao foi possivel localizar dados StatsBomb locais nem no fallback.")

print("\nResumo rapido:")
print(f"Players -> linhas: {len(df_players_local):,}, colunas: {df_players_local.shape[1]}")
print(f"Events  -> linhas: {len(df_events_local):,}, colunas: {df_events_local.shape[1]}")

# Insights focados no segundo dataset (StatsBomb)
event_col = "type.name" if "type.name" in df_events_local.columns else None
team_col = "team.name" if "team.name" in df_events_local.columns else None
player_col = "player.name" if "player.name" in df_events_local.columns else None

print("\nINSIGHTS - SEGUNDO DATASET (STATSBOMB)")
if event_col:
    print("Top 10 tipos de evento:")
    print(df_events_local[event_col].value_counts(dropna=False).head(10))
if team_col:
    print("\nTop 10 times por volume de evento:")
    print(df_events_local[team_col].value_counts(dropna=False).head(10))
if player_col:
    print("\nTop 10 jogadores por volume de evento:")
    print(df_events_local[player_col].value_counts(dropna=False).head(10))

if event_col and "shot.outcome.name" in df_events_local.columns:
    shots = df_events_local[df_events_local[event_col] == "Shot"]
    print(f"\nTotal de finalizacoes (Shot): {len(shots):,}")
    if len(shots) > 0:
        print("Resultado das finalizacoes:")
        print(shots["shot.outcome.name"].value_counts(dropna=False).head(10))

Raizes de busca:
- c:\Users\kaiki\OneDrive\Documentos\Hack_Esporte_Sorte\Data\Copper
- c:\Users\kaiki\OneDrive\Documentos\Hack_Esporte_Sorte\Data
- c:\Users\kaiki\OneDrive\Documentos\Hack_Esporte_Sorte
[LOCAL] Players nao encontrado no workspace. Usando Kaggle fallback.
[FALLBACK] Players carregado de: C:\Users\kaiki\.cache\kagglehub\datasets\hubertsidorowicz\football-players-stats-2025-2026\versions\30\players_data_light-2025_2026.csv
[LOCAL] StatsBomb nao encontrado no workspace. Usando Kaggle fallback.
[FALLBACK] JSONs de eventos encontrados: 3464

Resumo rapido:
Players -> linhas: 2,718, colunas: 53
Events  -> linhas: 116,004, colunas: 138

INSIGHTS - SEGUNDO DATASET (STATSBOMB)
Top 10 tipos de evento:
type.name
Pass             32954
Ball Receipt*    31301
Carry            27648
Pressure         10042
Ball Recovery     2593
Duel              1620
Block             1084
Dribble           1008
Clearance          977
Goal Keeper        899
Name: count, dtype: int64

Top 10 times por 

## Uso recomendado (baseado na documentacao dos PDFs)

Estrutura correta do StatsBomb Open Data:
1. `data/competitions.json`: catalogo de competicoes e temporadas.
2. `data/matches/{competition_id}/{season_id}.json`: lista de partidas da temporada.
3. `data/events/{match_id}.json`: eventos da partida (timeline).
4. `data/lineups/{match_id}.json`: escalações e jogadores.
5. `data/three-sixty/{match_id}.json`: freeze-frames 360 com `event_uuid`.

Melhor pratica:
- Sempre selecione primeiro competicao + temporada.
- Depois carregue partidas e somente os `match_id` necessarios.
- Faça join de eventos com 360 por `events.id == three_sixty.event_uuid`.
- Para modelo preditivo em jogo, crie features por janela temporal (ultimos 5-10 min).

In [11]:
# Loader canonico StatsBomb: competitions -> matches -> events/lineups/360
from pathlib import Path
import json
import pandas as pd
import kagglehub

# Descobrir raiz de dados
if 'path_sb' in globals():
    sb_root = Path(path_sb) / 'data'
else:
    sb_root = Path(kagglehub.dataset_download('saurabhshahane/statsbomb-football-data')) / 'data'

print('StatsBomb root:', sb_root)

# 1) Competitions
competitions_path = sb_root / 'competitions.json'
competitions = json.load(open(competitions_path, encoding='utf-8'))
df_comp = pd.json_normalize(competitions)
print('\nCompetitions loaded:', len(df_comp))
print(df_comp[['competition_id', 'competition_name', 'season_id', 'season_name']].head())

# Helpers
def load_matches(competition_id: int, season_id: int) -> pd.DataFrame:
    p = sb_root / 'matches' / str(competition_id) / f'{season_id}.json'
    if not p.exists():
        return pd.DataFrame()
    return pd.json_normalize(json.load(open(p, encoding='utf-8')))

def load_match_bundle(match_id: int, with_360: bool = True):
    e_path = sb_root / 'events' / f'{match_id}.json'
    l_path = sb_root / 'lineups' / f'{match_id}.json'
    t_path = sb_root / 'three-sixty' / f'{match_id}.json'

    events = pd.json_normalize(json.load(open(e_path, encoding='utf-8'))) if e_path.exists() else pd.DataFrame()
    lineups = pd.json_normalize(json.load(open(l_path, encoding='utf-8'))) if l_path.exists() else pd.DataFrame()

    three_sixty = pd.DataFrame()
    if with_360 and t_path.exists():
        three_sixty = pd.json_normalize(json.load(open(t_path, encoding='utf-8')))

    return events, lineups, three_sixty

# 2) Exemplo: selecionar uma competicao/temporada e carregar algumas partidas
example = df_comp[(df_comp['competition_name'] == 'La Liga') & (df_comp['season_name'] == '2020/2021')]
if example.empty:
    example = df_comp.head(1)

comp_id = int(example.iloc[0]['competition_id'])
season_id = int(example.iloc[0]['season_id'])
print(f"\nSelected competition/season: comp_id={comp_id}, season_id={season_id}")

df_matches = load_matches(comp_id, season_id)
print('Matches loaded:', len(df_matches))

if not df_matches.empty:
    sample_match_id = int(df_matches.iloc[0]['match_id'])
    ev, lu, t360 = load_match_bundle(sample_match_id, with_360=True)

    print(f"\nSample match_id: {sample_match_id}")
    print('Events:', ev.shape, '| Lineups:', lu.shape, '| 360:', t360.shape)

    # Join-chave recomendado: events.id == three_sixty.event_uuid
    if not ev.empty and not t360.empty and 'id' in ev.columns and 'event_uuid' in t360.columns:
        ev_360 = ev.merge(t360, left_on='id', right_on='event_uuid', how='left')
        coverage = ev_360['event_uuid'].notna().mean()
        print(f"360 coverage in events: {coverage:.2%}")


StatsBomb root: C:\Users\kaiki\.cache\kagglehub\datasets\saurabhshahane\statsbomb-football-data\versions\1378\data

Competitions loaded: 75
   competition_id        competition_name  season_id season_name
0               9           1. Bundesliga        281   2023/2024
1               9           1. Bundesliga         27   2015/2016
2            1267  African Cup of Nations        107        2023
3              16        Champions League          4   2018/2019
4              16        Champions League          1   2017/2018

Selected competition/season: comp_id=11, season_id=90
Matches loaded: 35

Sample match_id: 3773386
Events: (3891, 119) | Lineups: (2, 3) | 360: (0, 0)


## Previsao de gol nos proximos 10 minutos (baseline)

Objetivo: estimar a probabilidade de um time marcar gol nos proximos 10 minutos.

Abordagem:
1. Construir features por minuto e por time (eventos acumulados nos ultimos 10 min).
2. Gerar target binario: `1` se houver gol do time no intervalo `(t, t+10]`.
3. Treinar regressao logistica e avaliar com AUC e classification report.

In [12]:
# Construir base de treino para previsao de gol em 10 minutos
import json
import numpy as np
import pandas as pd

# Seleciona uma liga/temporada para evitar mistura de contextos
if 'df_comp' not in globals() or 'load_matches' not in globals():
    raise RuntimeError('Execute a celula do loader canonico antes desta.')

example = df_comp[(df_comp['competition_name'] == 'La Liga') & (df_comp['season_name'] == '2020/2021')]
if example.empty:
    example = df_comp.head(1)

comp_id = int(example.iloc[0]['competition_id'])
season_id = int(example.iloc[0]['season_id'])
matches_df = load_matches(comp_id, season_id)

if matches_df.empty:
    raise RuntimeError('Nao ha partidas para a competicao/temporada selecionada.')

# Para performance do notebook, usa uma amostra de partidas
MAX_MATCHES = 20
match_ids = matches_df['match_id'].dropna().astype(int).head(MAX_MATCHES).tolist()

rows = []
for mid in match_ids:
    ev_path = sb_root / 'events' / f'{mid}.json'
    if not ev_path.exists():
        continue

    ev = pd.json_normalize(json.load(open(ev_path, encoding='utf-8')))
    if ev.empty or 'minute' not in ev.columns or 'team.name' not in ev.columns or 'type.name' not in ev.columns:
        continue

    # Colunas auxiliares
    ev['minute'] = pd.to_numeric(ev['minute'], errors='coerce')
    ev = ev.dropna(subset=['minute', 'team.name', 'type.name'])
    ev['minute'] = ev['minute'].astype(int)

    # Marcadores de evento
    ev['is_shot'] = (ev['type.name'] == 'Shot').astype(int)
    ev['is_goal'] = ((ev['type.name'] == 'Shot') & (ev.get('shot.outcome.name', pd.Series(index=ev.index, dtype=object)) == 'Goal')).astype(int)
    ev['is_pass'] = (ev['type.name'] == 'Pass').astype(int)
    ev['is_pressure'] = (ev['type.name'] == 'Pressure').astype(int)
    ev['is_recovery'] = (ev['type.name'] == 'Ball Recovery').astype(int)

    teams = ev['team.name'].dropna().unique().tolist()
    max_minute = int(ev['minute'].max())

    for team in teams:
        tev = ev[ev['team.name'] == team].copy()

        # Pre-agregacao por minuto
        per_min = tev.groupby('minute')[['is_shot', 'is_goal', 'is_pass', 'is_pressure', 'is_recovery']].sum()

        for t in range(0, max_minute + 1):
            start = max(0, t - 9)
            hist = per_min.loc[start:t] if not per_min.empty else pd.DataFrame()

            shots_last10 = int(hist['is_shot'].sum()) if 'is_shot' in hist else 0
            passes_last10 = int(hist['is_pass'].sum()) if 'is_pass' in hist else 0
            pressure_last10 = int(hist['is_pressure'].sum()) if 'is_pressure' in hist else 0
            recovery_last10 = int(hist['is_recovery'].sum()) if 'is_recovery' in hist else 0
            goals_last10 = int(hist['is_goal'].sum()) if 'is_goal' in hist else 0

            # Target: gol do time em (t, t+10]
            future = per_min.loc[(per_min.index > t) & (per_min.index <= t + 10)] if not per_min.empty else pd.DataFrame()
            goal_next10 = int(future['is_goal'].sum() > 0) if 'is_goal' in future else 0

            rows.append({
                'match_id': mid,
                'team_name': team,
                'minute': t,
                'shots_last10': shots_last10,
                'passes_last10': passes_last10,
                'pressure_last10': pressure_last10,
                'recovery_last10': recovery_last10,
                'goals_last10': goals_last10,
                'goal_next10': goal_next10,
            })

df_goal10 = pd.DataFrame(rows)

print(f'Partidas usadas: {len(set(df_goal10["match_id"]))}')
print(f'Linhas de treino: {len(df_goal10):,}')
print('Taxa de positivos (goal_next10=1):', round(df_goal10['goal_next10'].mean(), 4))
print(df_goal10.head())

Partidas usadas: 20
Linhas de treino: 3,786
Taxa de positivos (goal_next10=1): 0.1358
   match_id         team_name  minute  shots_last10  passes_last10  pressure_last10  recovery_last10  goals_last10  goal_next10
0   3773386  Deportivo Alavés       0             0              2                9                0             0            0
1   3773386  Deportivo Alavés       1             0              4               12                0             0            0
2   3773386  Deportivo Alavés       2             0              9               15                0             0            0
3   3773386  Deportivo Alavés       3             0             18               18                1             0            0
4   3773386  Deportivo Alavés       4             0             29               18                1             0            0


In [13]:
# Treinar modelo baseline (regressao logistica) e criar funcao de inferencia
import sys
import subprocess

try:
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, classification_report
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn'])
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, classification_report

feature_cols = ['minute', 'shots_last10', 'passes_last10', 'pressure_last10', 'recovery_last10', 'goals_last10']
X = df_goal10[feature_cols].copy()
y = df_goal10['goal_next10'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

goal10_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

goal10_model.fit(X_train, y_train)
proba = goal10_model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba)
print(f'AUC (goal em 10 min): {auc:.4f}')
print('\nClassification report:')
print(classification_report(y_test, pred, digits=3))

# Importancia aproximada (coeficientes da regressao)
coefs = goal10_model.named_steps['clf'].coef_[0]
importance = pd.Series(coefs, index=feature_cols).sort_values(key=np.abs, ascending=False)
print('\nTop sinais do modelo (|coef|):')
print(importance)

def predict_goal_next10_prob(minute, shots_last10, passes_last10, pressure_last10, recovery_last10, goals_last10):
    row = pd.DataFrame([{
        'minute': minute,
        'shots_last10': shots_last10,
        'passes_last10': passes_last10,
        'pressure_last10': pressure_last10,
        'recovery_last10': recovery_last10,
        'goals_last10': goals_last10,
    }])
    p = float(goal10_model.predict_proba(row)[0, 1])
    return p

# Exemplo de uso (cenario ficticio)
p_demo = predict_goal_next10_prob(
    minute=62,
    shots_last10=3,
    passes_last10=45,
    pressure_last10=8,
    recovery_last10=5,
    goals_last10=0,
)
print(f'\nExemplo -> Probabilidade de gol nos proximos 10 min: {p_demo:.2%}')

AUC (goal em 10 min): 0.5694

Classification report:
              precision    recall  f1-score   support

           0      0.876     0.502     0.639       818
           1      0.149     0.550     0.234       129

    accuracy                          0.509       947
   macro avg      0.512     0.526     0.436       947
weighted avg      0.777     0.509     0.584       947


Top sinais do modelo (|coef|):
minute            -0.175006
passes_last10      0.159548
goals_last10      -0.156058
recovery_last10   -0.058339
shots_last10       0.057063
pressure_last10    0.034902
dtype: float64

Exemplo -> Probabilidade de gol nos proximos 10 min: 46.95%


## Caso de uso: sequencia de vitorias/derrotas do Manchester City

Objetivo: identificar a sequencia recente de resultados do Manchester City (V, E, D), com contagem de serie atual.

In [14]:
# Sequencia de resultados do Manchester City na temporada mais recente disponivel
import json
from pathlib import Path
import pandas as pd

if 'sb_root' not in globals():
    raise RuntimeError('Execute a celula do loader canonico antes desta.')

team_name = 'Manchester City'
matches_root = Path(sb_root) / 'matches'

all_rows = []
for season_file in matches_root.rglob('*.json'):
    try:
        season_matches = json.load(open(season_file, encoding='utf-8'))
    except Exception:
        continue

    for m in season_matches:
        home = m.get('home_team', {}).get('home_team_name')
        away = m.get('away_team', {}).get('away_team_name')

        if team_name not in [home, away]:
            continue

        home_score = m.get('home_score')
        away_score = m.get('away_score')
        if home_score is None or away_score is None:
            continue

        is_home = (home == team_name)
        gf = home_score if is_home else away_score
        ga = away_score if is_home else home_score

        if gf > ga:
            result = 'V'
        elif gf < ga:
            result = 'D'
        else:
            result = 'E'

        all_rows.append({
            'match_id': m.get('match_id'),
            'date': m.get('match_date'),
            'competition': m.get('competition', {}).get('competition_name'),
            'season': m.get('season', {}).get('season_name'),
            'opponent': away if is_home else home,
            'home_away': 'Casa' if is_home else 'Fora',
            'gf': gf,
            'ga': ga,
            'result': result,
        })

city = pd.DataFrame(all_rows)

if city.empty:
    print('Nenhuma partida do Manchester City foi encontrada no dataset.')
else:
    city['date'] = pd.to_datetime(city['date'], errors='coerce')
    city = city.sort_values('date').reset_index(drop=True)

    # Temporada mais recente disponivel para o time
    latest_season = city.iloc[-1]['season']
    city_season = city[city['season'] == latest_season].copy().sort_values('date').reset_index(drop=True)

    # Sequencia da temporada
    seq = ''.join(city_season['result'].tolist())

    # Serie atual da temporada
    last_result = city_season.iloc[-1]['result']
    streak_len = 1
    for r in city_season['result'].iloc[:-1][::-1]:
        if r == last_result:
            streak_len += 1
        else:
            break

    legenda = {'V': 'Vitorias', 'D': 'Derrotas', 'E': 'Empates'}

    print(f"Temporada mais recente disponivel no dataset para {team_name}: {latest_season}")
    print(f"Total de jogos na temporada: {len(city_season)}")
    print(f"Sequencia (cronologica): {seq}")
    print(f"Serie atual: {streak_len} {legenda[last_result]}")

    print('\nJogos da temporada:')
    print(city_season[['date', 'competition', 'season', 'opponent', 'home_away', 'gf', 'ga', 'result']])

    # Observacao sobre disponibilidade
    print('\nObs: este resultado depende das temporadas disponiveis no StatsBomb Open Data, nao necessariamente da temporada atual real.')

Temporada mais recente disponivel no dataset para Manchester City: 2015/2016
Total de jogos na temporada: 38
Sequencia (cronologica): VVVVVDDVVEVEDVDVDVEVEVEVDDDVEDVVVEVDEE
Serie atual: 2 Empates

Jogos da temporada:
         date     competition     season              opponent home_away  gf  ga result
0  2015-08-10  Premier League  2015/2016  West Bromwich Albion      Fora   3   0      V
1  2015-08-16  Premier League  2015/2016               Chelsea      Casa   3   0      V
2  2015-08-23  Premier League  2015/2016               Everton      Fora   2   0      V
3  2015-08-29  Premier League  2015/2016               Watford      Casa   2   0      V
4  2015-09-12  Premier League  2015/2016        Crystal Palace      Fora   1   0      V
5  2015-09-19  Premier League  2015/2016       West Ham United      Casa   1   2      D
6  2015-09-26  Premier League  2015/2016     Tottenham Hotspur      Fora   1   4      D
7  2015-10-03  Premier League  2015/2016      Newcastle United      Casa   6   

## Recorte de dados atuais (temporadas recentes)

Como o Open Data nao cobre todos os times em todas as temporadas recentes, este bloco:
1. considera apenas temporadas com ano final >= 2023;
2. mostra competicoes ativas no recorte atual;
3. responde a pergunta de sequencia apenas se houver cobertura atual do time.

In [15]:
# Analise de cobertura atual e consulta de sequencia no recorte recente
import re
import json
from pathlib import Path
import pandas as pd

if 'df_comp' not in globals() or 'sb_root' not in globals():
    raise RuntimeError('Execute a celula do loader canonico antes desta.')

def season_end_year(season_name: str):
    if not isinstance(season_name, str):
        return None
    years = [int(y) for y in re.findall(r'(?:19|20)\d{2}', season_name)]
    if not years:
        return None
    return max(years)

comp_recent = df_comp.copy()
comp_recent['season_end_year'] = comp_recent['season_name'].apply(season_end_year)

CURRENT_CUTOFF = 2023
comp_recent = comp_recent[comp_recent['season_end_year'].fillna(0) >= CURRENT_CUTOFF].copy()
comp_recent = comp_recent.sort_values(['season_end_year', 'competition_name'], ascending=[False, True])

print(f'Temporadas consideradas como atuais: fim >= {CURRENT_CUTOFF}')
print(f'Competicao/temporada disponiveis nesse recorte: {len(comp_recent)}')
print(comp_recent[['competition_name', 'season_name', 'season_end_year']].drop_duplicates().head(20))

# Monta base de partidas do recorte recente
rows = []
for _, row in comp_recent[['competition_id', 'season_id', 'competition_name', 'season_name']].drop_duplicates().iterrows():
    p = Path(sb_root) / 'matches' / str(int(row['competition_id'])) / f"{int(row['season_id'])}.json"
    if not p.exists():
        continue
    try:
        matches = json.load(open(p, encoding='utf-8'))
    except Exception:
        continue

    for m in matches:
        rows.append({
            'match_id': m.get('match_id'),
            'date': m.get('match_date'),
            'competition': row['competition_name'],
            'season': row['season_name'],
            'home_team': m.get('home_team', {}).get('home_team_name'),
            'away_team': m.get('away_team', {}).get('away_team_name'),
            'home_score': m.get('home_score'),
            'away_score': m.get('away_score'),
        })

recent_matches = pd.DataFrame(rows)
print(f'\nPartidas no recorte atual: {len(recent_matches)}')

if recent_matches.empty:
    print('Nao ha partidas no recorte atual no dataset.')
else:
    recent_matches['date'] = pd.to_datetime(recent_matches['date'], errors='coerce')

    # Times disponiveis no recorte
    teams_recent = pd.unique(pd.concat([
        recent_matches['home_team'].dropna(),
        recent_matches['away_team'].dropna()
    ], ignore_index=True))

    print(f'Times unicos no recorte atual: {len(teams_recent)}')

    # Consulta especifica
    team_name = 'Manchester City'

    team_games = recent_matches[(recent_matches['home_team'] == team_name) | (recent_matches['away_team'] == team_name)].copy()

    if team_games.empty:
        print(f"\n{team_name} nao possui jogos no recorte atual (>= {CURRENT_CUTOFF}) neste Open Data.")
        # Sugestao de times com dados atuais
        sample_teams = sorted(list(teams_recent))[:25]
        print('Exemplos de times com cobertura atual no dataset:')
        print(sample_teams)
    else:
        # Calcula resultado do ponto de vista do time
        def _res(r):
            is_home = r['home_team'] == team_name
            gf = r['home_score'] if is_home else r['away_score']
            ga = r['away_score'] if is_home else r['home_score']
            if gf > ga:
                return 'V'
            if gf < ga:
                return 'D'
            return 'E'

        team_games = team_games.sort_values('date').reset_index(drop=True)
        team_games['result'] = team_games.apply(_res, axis=1)
        seq = ''.join(team_games['result'].tolist())

        last = team_games.iloc[-1]['result']
        streak = 1
        for r in team_games['result'].iloc[:-1][::-1]:
            if r == last:
                streak += 1
            else:
                break

        legenda = {'V': 'Vitorias', 'D': 'Derrotas', 'E': 'Empates'}
        print(f"\n{team_name} no recorte atual:")
        print(f"Jogos: {len(team_games)} | Sequencia: {seq} | Serie atual: {streak} {legenda[last]}")
        print(team_games[['date', 'competition', 'season', 'home_team', 'away_team', 'home_score', 'away_score', 'result']].tail(10))

Temporadas consideradas como atuais: fim >= 2023
Competicao/temporada disponiveis nesse recorte: 8
          competition_name season_name  season_end_year
71       UEFA Women's Euro        2025             2025
0            1. Bundesliga   2023/2024             2024
21            Copa America        2024             2024
68               UEFA Euro        2024             2024
2   African Cup of Nations        2023             2023
58                 Ligue 1   2022/2023             2023
61     Major League Soccer        2023             2023
73       Women's World Cup        2023             2023

Partidas no recorte atual: 302
Times unicos no recorte atual: 146

Manchester City nao possui jogos no recorte atual (>= 2023) neste Open Data.
Exemplos de times com cobertura atual no dataset:
['AC Ajaccio', 'AS Monaco', 'Albania', 'Algeria', 'Angers', 'Angola', 'Argentina', "Argentina Women's", 'Augsburg', "Australia Women's", 'Austria', 'Auxerre', 'Bayer Leverkusen', 'Bayern Munich', 'Belgi

## Analise de jogos: Bayern Munich (recorte atual)

Consulta focada no Bayern Munich usando o mesmo recorte de temporadas recentes (ano final >= 2023).

In [16]:
# Sequencia e resumo de jogos do Bayern Munich com diagnostico de cobertura
import pandas as pd

if 'recent_matches' not in globals() or recent_matches.empty:
    raise RuntimeError('Execute primeiro a celula de recorte atual para criar recent_matches.')

if 'sb_root' not in globals() or 'load_matches' not in globals() or 'df_comp' not in globals():
    raise RuntimeError('Execute a celula do loader canonico antes desta.')

team_name = 'Bayern Munich'

# 1) Recorte atual (>= 2023)
tg_recent = recent_matches[(recent_matches['home_team'] == team_name) | (recent_matches['away_team'] == team_name)].copy()

print('=== Recorte atual (temporadas recentes) ===')
print(f'Jogos do {team_name} no recorte atual: {len(tg_recent)}')

# 2) Diagnostico de cobertura historica por temporada para o time
coverage_rows = []
for _, row in df_comp[['competition_id', 'season_id', 'competition_name', 'season_name']].drop_duplicates().iterrows():
    mdf = load_matches(int(row['competition_id']), int(row['season_id']))
    if mdf.empty or 'home_team.home_team_name' not in mdf.columns or 'away_team.away_team_name' not in mdf.columns:
        continue

    tdf = mdf[(mdf['home_team.home_team_name'] == team_name) | (mdf['away_team.away_team_name'] == team_name)].copy()
    if tdf.empty:
        continue

    coverage_rows.append({
        'competition_name': row['competition_name'],
        'season_name': row['season_name'],
        'games': len(tdf),
        'competition_id': int(row['competition_id']),
        'season_id': int(row['season_id']),
    })

coverage = pd.DataFrame(coverage_rows).sort_values(['games', 'competition_name', 'season_name'], ascending=[False, True, True]).reset_index(drop=True)

if coverage.empty:
    print(f'Nao ha cobertura de partidas para {team_name} neste dataset.')
else:
    print('\n=== Cobertura por temporada (top 10) ===')
    print(coverage.head(10))

    # Escolhe temporada com maior cobertura para responder com mais robustez
    best = coverage.iloc[0]
    comp_id = int(best['competition_id'])
    season_id = int(best['season_id'])
    comp_name = best['competition_name']
    season_name = best['season_name']

    mdf = load_matches(comp_id, season_id)
    tg = mdf[(mdf['home_team.home_team_name'] == team_name) | (mdf['away_team.away_team_name'] == team_name)].copy()

    tg['date'] = pd.to_datetime(tg['match_date'], errors='coerce')
    tg = tg.sort_values('date').reset_index(drop=True)

    def result_for_team(r):
        is_home = (r['home_team.home_team_name'] == team_name)
        gf = r['home_score'] if is_home else r['away_score']
        ga = r['away_score'] if is_home else r['home_score']
        if gf > ga:
            return 'V'
        if gf < ga:
            return 'D'
        return 'E'

    tg['result'] = tg.apply(result_for_team, axis=1)
    tg['gf'] = tg.apply(lambda r: r['home_score'] if r['home_team.home_team_name'] == team_name else r['away_score'], axis=1)
    tg['ga'] = tg.apply(lambda r: r['away_score'] if r['home_team.home_team_name'] == team_name else r['home_score'], axis=1)
    tg['opponent'] = tg.apply(lambda r: r['away_team.away_team_name'] if r['home_team.home_team_name'] == team_name else r['home_team.home_team_name'], axis=1)
    tg['home_away'] = tg.apply(lambda r: 'Casa' if r['home_team.home_team_name'] == team_name else 'Fora', axis=1)

    seq = ''.join(tg['result'].tolist())
    last = tg.iloc[-1]['result']
    streak = 1
    for r in tg['result'].iloc[:-1][::-1]:
        if r == last:
            streak += 1
        else:
            break

    legenda = {'V': 'Vitorias', 'D': 'Derrotas', 'E': 'Empates'}

    wins = int((tg['result'] == 'V').sum())
    draws = int((tg['result'] == 'E').sum())
    losses = int((tg['result'] == 'D').sum())
    goals_for = int(tg['gf'].sum())
    goals_against = int(tg['ga'].sum())
    points = wins * 3 + draws
    ppg = points / len(tg) if len(tg) else 0

    print('\n=== Melhor temporada disponivel para analise (maior cobertura) ===')
    print(f'Competicao: {comp_name} | Temporada: {season_name} | Jogos: {len(tg)}')
    print(f'Sequencia (cronologica): {seq}')
    print(f'Serie atual: {streak} {legenda[last]}')
    print(f'Resumo: V={wins}, E={draws}, D={losses}, GP={goals_for}, GC={goals_against}, P/J={ppg:.2f}')

    print('\nUltimos 10 jogos dessa temporada:')
    print(tg[['date', 'opponent', 'home_away', 'gf', 'ga', 'result']].tail(10))

    print('\nObs: no recorte atual (>= 2023), a cobertura para Bayern esta limitada neste Open Data.')

=== Recorte atual (temporadas recentes) ===
Jogos do Bayern Munich no recorte atual: 2

=== Cobertura por temporada (top 10) ===
     competition_name season_name  games  competition_id  season_id
0       1. Bundesliga   2015/2016     34               9         27
1       1. Bundesliga   2023/2024      2               9        281
2    Champions League   2009/2010      1              16         21
3    Champions League   2011/2012      1              16         23
4    Champions League   2012/2013      1              16         24
5  UEFA Europa League   1988/1989      1              35         75

=== Melhor temporada disponivel para analise (maior cobertura) ===
Competicao: 1. Bundesliga | Temporada: 2015/2016 | Jogos: 34
Sequencia (cronologica): VVVVVVVVVVEVVVDVVVVEVVVDEVVVVVVEVV
Serie atual: 2 Vitorias
Resumo: V=28, E=4, D=2, GP=80, GC=17, P/J=2.59

Ultimos 10 jogos dessa temporada:
         date                  opponent home_away  gf  ga result
24 2016-03-05         Borussia Dort

## Funcao reutilizavel para o assistente (qualquer time)

A funcao `analisar_time` responde no mesmo formato para qualquer equipe, com dois modos:
1. `atual`: usa apenas temporadas recentes (ano final >= cutoff).
2. `maior_cobertura`: usa a temporada com mais jogos disponiveis para o time.

In [17]:
# Funcao generica para analise de sequencia de resultados
import re
import json
import pandas as pd
from pathlib import Path

if 'df_comp' not in globals() or 'sb_root' not in globals() or 'load_matches' not in globals():
    raise RuntimeError('Execute a celula do loader canonico antes desta.')


def _season_end_year(season_name: str):
    if not isinstance(season_name, str):
        return None
    years = [int(y) for y in re.findall(r'(?:19|20)\d{2}', season_name)]
    return max(years) if years else None


def _compute_team_frame(team_df: pd.DataFrame, team_name: str) -> pd.DataFrame:
    tdf = team_df.copy()
    tdf['date'] = pd.to_datetime(tdf['date'], errors='coerce')
    tdf = tdf.sort_values('date').reset_index(drop=True)

    def _res(r):
        is_home = r['home_team'] == team_name
        gf = r['home_score'] if is_home else r['away_score']
        ga = r['away_score'] if is_home else r['home_score']
        if gf > ga:
            return 'V'
        if gf < ga:
            return 'D'
        return 'E'

    tdf['result'] = tdf.apply(_res, axis=1)
    tdf['gf'] = tdf.apply(lambda r: r['home_score'] if r['home_team'] == team_name else r['away_score'], axis=1)
    tdf['ga'] = tdf.apply(lambda r: r['away_score'] if r['home_team'] == team_name else r['home_score'], axis=1)
    tdf['opponent'] = tdf.apply(lambda r: r['away_team'] if r['home_team'] == team_name else r['home_team'], axis=1)
    tdf['home_away'] = tdf.apply(lambda r: 'Casa' if r['home_team'] == team_name else 'Fora', axis=1)
    return tdf


def _summary_from_frame(tdf: pd.DataFrame):
    seq = ''.join(tdf['result'].tolist())
    last = tdf.iloc[-1]['result']
    streak = 1
    for r in tdf['result'].iloc[:-1][::-1]:
        if r == last:
            streak += 1
        else:
            break

    wins = int((tdf['result'] == 'V').sum())
    draws = int((tdf['result'] == 'E').sum())
    losses = int((tdf['result'] == 'D').sum())
    goals_for = int(tdf['gf'].sum())
    goals_against = int(tdf['ga'].sum())
    points = wins * 3 + draws
    ppg = points / len(tdf) if len(tdf) else 0.0

    return {
        'sequence': seq,
        'last_result': last,
        'streak_len': streak,
        'wins': wins,
        'draws': draws,
        'losses': losses,
        'goals_for': goals_for,
        'goals_against': goals_against,
        'ppg': ppg,
    }


def analisar_time(team_name: str, mode: str = 'atual', current_cutoff: int = 2023, tail_n: int = 10):
    # Monta base mestre de partidas do dataset
    rows = []
    comp_df = df_comp.copy()
    comp_df['season_end_year'] = comp_df['season_name'].apply(_season_end_year)

    if mode == 'atual':
        comp_df = comp_df[comp_df['season_end_year'].fillna(0) >= current_cutoff].copy()

    for _, row in comp_df[['competition_id', 'season_id', 'competition_name', 'season_name', 'season_end_year']].drop_duplicates().iterrows():
        mdf = load_matches(int(row['competition_id']), int(row['season_id']))
        if mdf.empty:
            continue

        required = ['match_id', 'match_date', 'home_team.home_team_name', 'away_team.away_team_name', 'home_score', 'away_score']
        if any(c not in mdf.columns for c in required):
            continue

        for _, m in mdf[required].iterrows():
            rows.append({
                'match_id': m['match_id'],
                'date': m['match_date'],
                'competition': row['competition_name'],
                'season': row['season_name'],
                'season_end_year': row['season_end_year'],
                'home_team': m['home_team.home_team_name'],
                'away_team': m['away_team.away_team_name'],
                'home_score': m['home_score'],
                'away_score': m['away_score'],
            })

    base = pd.DataFrame(rows)
    if base.empty:
        print('Base de partidas vazia para o modo selecionado.')
        return None

    team_games = base[(base['home_team'] == team_name) | (base['away_team'] == team_name)].copy()
    if team_games.empty:
        print(f'{team_name} nao possui jogos no modo={mode}.')
        return None

    if mode == 'maior_cobertura':
        cov = (team_games.groupby(['competition', 'season', 'season_end_year'])
               .size().reset_index(name='games')
               .sort_values(['games', 'season_end_year'], ascending=[False, False]))
        chosen = cov.iloc[0]
        team_games = team_games[(team_games['competition'] == chosen['competition']) & (team_games['season'] == chosen['season'])].copy()

    team_games = _compute_team_frame(team_games, team_name)
    summary = _summary_from_frame(team_games)
    legenda = {'V': 'Vitorias', 'D': 'Derrotas', 'E': 'Empates'}

    print(f'Equipe: {team_name} | modo={mode}')
    if mode == 'atual':
        print(f'Recorte: temporadas com fim >= {current_cutoff}')
    print(f"Jogos: {len(team_games)}")
    print(f"Sequencia: {summary['sequence']}")
    print(f"Serie atual: {summary['streak_len']} {legenda[summary['last_result']]}")
    print(f"Resumo: V={summary['wins']}, E={summary['draws']}, D={summary['losses']}, GP={summary['goals_for']}, GC={summary['goals_against']}, P/J={summary['ppg']:.2f}")

    print('\nUltimos jogos:')
    print(team_games[['date', 'competition', 'season', 'opponent', 'home_away', 'gf', 'ga', 'result']].tail(tail_n))

    return {
        'team': team_name,
        'mode': mode,
        'games': len(team_games),
        **summary,
        'table': team_games,
    }

In [18]:
# Exemplos prontos para o assistente
_ = analisar_time('Bayern Munich', mode='atual', current_cutoff=2023, tail_n=10)

print('\n' + '='*80 + '\n')

_ = analisar_time('Bayern Munich', mode='maior_cobertura', tail_n=10)

print('\n' + '='*80 + '\n')

_ = analisar_time('Manchester City', mode='atual', current_cutoff=2023, tail_n=10)

Equipe: Bayern Munich | modo=atual
Recorte: temporadas com fim >= 2023
Jogos: 2
Sequencia: ED
Serie atual: 1 Derrotas
Resumo: V=0, E=1, D=1, GP=2, GC=5, P/J=0.50

Ultimos jogos:
        date    competition     season          opponent home_away  gf  ga result
0 2023-09-15  1. Bundesliga  2023/2024  Bayer Leverkusen      Casa   2   2      E
1 2024-02-10  1. Bundesliga  2023/2024  Bayer Leverkusen      Fora   0   3      D


Equipe: Bayern Munich | modo=maior_cobertura
Jogos: 34
Sequencia: VVVVVVVVVVEVVVDVVVVEVVVDEVVVVVVEVV
Serie atual: 2 Vitorias
Resumo: V=28, E=4, D=2, GP=80, GC=17, P/J=2.59

Ultimos jogos:
         date    competition     season                  opponent home_away  gf  ga result
24 2016-03-05  1. Bundesliga  2015/2016         Borussia Dortmund      Fora   0   0      E
25 2016-03-12  1. Bundesliga  2015/2016             Werder Bremen      Casa   5   0      V
26 2016-03-19  1. Bundesliga  2015/2016                   FC Köln      Fora   1   0      V
27 2016-04-02  1. Bund

In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Filtrar a Serie A (Sua especialidade)
# Assumindo que seu DataFrame principal se chama 'df'
df_serie_a = df[df['Comp'] == 'it Serie A'].copy()

# 2. Seleção Dinâmica de Métricas (Evita o KeyError)
metrics_desejadas = ['Gls', 'Ast', 'Sh', 'SoT', 'xG', 'npxG', '90s']
# Só pegamos o que realmente existe no dataset (especialmente se for a versão 'light')
metrics = [m for m in metrics_desejadas if m in df_serie_a.columns]

print(f"✅ Métricas encontradas para a Aura: {metrics}")

# 3. Cálculo da Aura (Média Móvel dos últimos 5 jogos)
# Ordenamos por time e idade/rank para criar uma sequência lógica
df_serie_a = df_serie_a.sort_values(['Squad', 'Age']) 

rolling = df_serie_a.groupby('Squad')[metrics].rolling(window=5, min_periods=1).mean().shift(1).reset_index(level=0, drop=True)

# Adicionando as colunas de Aura ao DataFrame
df_serie_a[[f'{m}_aura' for m in metrics]] = rolling

# 4. Salvar na SILVER com o motor 'fastparquet'
os.makedirs('Data/Silver', exist_ok=True)

try:
    # Especificamos o engine='fastparquet' para evitar o erro do Arrow
    df_serie_a.to_parquet('Data/Silver/serie_a_ready.parquet', engine='fastparquet', index=False)
    print("💎 Silver Layer pronta e salva em Parquet! O Pudim já pode comemorar 🐶.")
except Exception as e:
    print(f"❌ Erro ao salvar Parquet: {e}")
    # Se tudo falhar, o Pickle salva sua vida no hackathon
    df_serie_a.to_pickle('Data/Silver/serie_a_ready.pkl')
    print("⚠️ Salvo como Pickle (emergência) para você não travar.")

✅ Métricas encontradas para a Aura: ['Gls', 'Ast', 'Sh', 'SoT', '90s']
❌ Erro ao salvar Parquet: [Errno 2] No such file or directory: 'Data/Cooper/serie_a_ready.parquet'
⚠️ Salvo como Pickle (emergência) para você não travar.
